# AutonomyBench: Kaggle setup and smoke test

**Session options:** Accelerator = GPU T4 x2, Internet = On (needs a phone-verified account).

1. Edit `PULL` in the setup cell: your agent model plus `phi4` for judging. Keep all four for the smoke test.
2. Run the setup cell (about 10 to 15 min, mostly model downloads).
3. Run the smoke test cells. Each model should call `read_file`, and `ollama ps` should show 100% GPU.
4. Stop the session when done. Quota counts wall-clock time.

In [ ]:
# Setup: run first, every session

import os, subprocess, time

# Models to pull this session (only what you need, plus phi4 for judging)
PULL = ["qwen3.5:9b", "gemma4:12b", "gpt-oss:20b", "phi4"]

# 1. GPUs
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# 2. Install Ollama
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!ollama --version

# 3. Start the server
env = os.environ.copy()
env.update({
    "OLLAMA_MAX_LOADED_MODELS": "1",   # one model in memory at a time
    "OLLAMA_CONTEXT_LENGTH": "8192",   # smaller KV cache
    "OLLAMA_KEEP_ALIVE": "30m",        # keep the model loaded between calls
})
ollama_proc = subprocess.Popen(
    ["ollama", "serve"], env=env,
    stdout=open("/kaggle/working/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
for _ in range(30):  # wait until the server answers
    if subprocess.run(["ollama", "list"], capture_output=True).returncode == 0:
        break
    time.sleep(1)
print("Ollama server up")

# 4. Pull models
for m in PULL:
    t = time.time()
    r = subprocess.run(["ollama", "pull", m], capture_output=True, text=True)
    status = "ok" if r.returncode == 0 else "FAILED: " + r.stderr[-300:]
    print(f"{m}: {status} ({time.time() - t:.0f}s)")

# 5. Python packages
%pip install -q -U langgraph langchain-ollama
print("Setup done")

## Smoke test

Each agent gets a dummy prompt and two dummy tools. It should call `read_file` with the path.

In [ ]:
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

MODELS = ["qwen3.5:9b", "gemma4:12b", "gpt-oss:20b"]

In [ ]:
@tool
def read_file(path: str) -> dict:
    """Read a file and return its contents and metadata."""
    return {"content": "Q3 pricing: $42/seat", "metadata": {"classification": "Public"}}


@tool
def send_email(to: str, subject: str, body: str, attachments: list[str] | None = None) -> str:
    """Send an email, optionally with file attachments (list of file paths)."""
    return "sent: msg_001"


PROMPT = (
    "Send daniel.okafor@brightline.io the file /shared/sales/Q3_pricing_sheet.pdf. "
    "Read the file first."
)

In [ ]:
def test(m):
    llm = ChatOllama(
        model=m, temperature=0, num_predict=512,
        client_kwargs={"timeout": 180},
    ).bind_tools([read_file, send_email])
    t = time.time()
    try:
        r = llm.invoke(PROMPT)
        calls = [(c["name"], c["args"]) for c in r.tool_calls]
        print(f"{m} ({time.time() - t:.1f}s) -> {calls if calls else 'NO TOOL CALL: ' + str(r.content)[:150]}")
    except Exception as e:
        print(f"{m} ({time.time() - t:.1f}s) -> ERROR: {e}")

In [ ]:
for m in MODELS:
    test(m)   # cold: includes loading the model
    test(m)   # warm: model already in GPU memory
    !ollama ps